# Clean up

Removes everything this pattern deployed, so the project goes back to how it started.

Nothing is deleted until you type `YES` in the box that pops up when you run the next cell. Any other answer is a dry run: every step only prints what it would remove. Each step is safe to run again - a partially deployed or already-cleaned project will not raise.

**Stop these first:**

1. Stop the feed: press **Stop feed** in `integrations_tutorial.ipynb`, or shut that notebook's kernel down.
2. Stop the extractor: press **Ctrl+C** in the terminal where `run-extractor.ps1` is running.

A running extractor keeps checking in and writing data, so it would be working against the cleanup.

In [11]:
import sys

sys.path.insert(0, "..")

from helpers import feed, ops
from helpers.auth import get_client
from helpers.constants import (
    LOCATION_FILTER_EXTERNAL_ID,
    INTEGRATION_EXTERNAL_ID,
    RAW_DATABASE,
    RAW_TABLE,
    SPACE,
    TRANSFORMATION_EXTERNAL_IDS,
    WORKFLOW_EXTERNAL_ID,
    WORKFLOW_VERSION,
)

client = get_client()
print(f"Target project: {client.config.project}")

# A box pops up at the top of the window. Only an exact YES deletes anything.
answer = input(
    f"This deletes what integrations_tutorial.ipynb deployed to project '{client.config.project}'. "
    "Type YES (capitals) to continue, anything else for a dry run: "
)
CONFIRM = answer.strip() == "YES"
print("Confirmed - resources WILL be deleted." if CONFIRM else "Not confirmed. Dry run: nothing will be deleted.")

Target project: cdf-sandbox-controltower
Confirmed - resources WILL be deleted.


## 1. Workflow

Trigger first, then the version, then the workflow itself.

In [12]:
import time

from cognite.client.exceptions import CogniteAPIError

triggers = [t for t in client.workflows.triggers.list(limit=None) if t.external_id.startswith(WORKFLOW_EXTERNAL_ID)]
print(f"Triggers      : {[t.external_id for t in triggers]}")
print(f"Workflow      : {WORKFLOW_EXTERNAL_ID} version {WORKFLOW_VERSION}")

if CONFIRM:
    for trigger in triggers:
        client.workflows.triggers.delete(trigger.external_id)
        print(f"  deleted trigger {trigger.external_id}")
    # A workflow cannot be deleted while it has a running execution. Cancelling takes a moment to
    # take effect (the transformation underneath has to wind down), so cancel, then retry the delete.
    client.workflows.versions.delete((WORKFLOW_EXTERNAL_ID, WORKFLOW_VERSION), ignore_unknown_ids=True)
    for attempt in range(40):
        for execution in client.workflows.executions.list(statuses=["running"], limit=None):
            if execution.workflow_external_id == WORKFLOW_EXTERNAL_ID:
                client.workflows.executions.cancel(execution.id, reason="cleanup")
                print(f"  cancelled running execution {execution.id}")
        try:
            client.workflows.delete(WORKFLOW_EXTERNAL_ID, ignore_unknown_ids=True)
            break
        except CogniteAPIError as error:
            if "running executions" not in str(error) or attempt == 39:
                raise
            print("  waiting for the last execution to stop...")
            time.sleep(5)
    print("  deleted workflow and version")

Triggers      : ['wf_psp_pump_processing_schedule']
Workflow      : wf_psp_pump_processing version v1
  deleted trigger wf_psp_pump_processing_schedule
  deleted workflow and version


## 2. Transformations

In [13]:
print(f"Transformations: {TRANSFORMATION_EXTERNAL_IDS}")

if CONFIRM:
    for external_id in TRANSFORMATION_EXTERNAL_IDS:
        client.transformations.schedules.delete(external_id=external_id, ignore_unknown_ids=True)
    client.transformations.delete(external_id=TRANSFORMATION_EXTERNAL_IDS, ignore_unknown_ids=True)
    print("  deleted transformations and any schedules")

Transformations: ['tr_psp_pump_assets', 'tr_psp_pump_timeseries', 'tr_psp_pump_datapoints']
  deleted transformations and any schedules


## 3. Location filter

The **Pump House** filter that made the data show up in Search. It goes before the data, so Search never points at a space that is being emptied.

In [14]:
import httpx

location_url = f"{client.config.base_url}/apps/v1/projects/{client.config.project}/storage/config/locationfilters"
filters = [
    item
    for item in client.post(f"{location_url}/list", json={}).json().get("items", [])
    if item.get("externalId") == LOCATION_FILTER_EXTERNAL_ID
]
print(f"Location filter: {[item['name'] for item in filters] or 'none found'}")

if CONFIRM:
    for item in filters:
        # The SDK client has get/post/put but no delete, so send this one request directly.
        header, value = client.config.credentials.authorization_header()
        httpx.delete(f"{location_url}/{item['id']}", headers={header: value}).raise_for_status()
        print(f"  deleted location filter {item['name']}")

Location filter: ['Pump House']
  deleted location filter Pump House


## 4. Data model instances, then the space

A space cannot be deleted while it still holds instances, so the instances go first.
Deleting a `CogniteTimeSeries` instance removes its datapoints with it.

In [15]:
instances = client.data_modeling.instances.list(space=SPACE, limit=None)
print(f"Instances in {SPACE}: {len(instances)}")

if CONFIRM:
    if instances:
        client.data_modeling.instances.delete(nodes=[(i.space, i.external_id) for i in instances])
        print(f"  deleted {len(instances)} instances")
    client.data_modeling.spaces.delete(SPACE)
    print(f"  deleted space {SPACE}")

Instances in sp_psp_integrations: 13
  deleted 13 instances
  deleted space sp_psp_integrations


## 5. RAW

In [16]:
# The RAW delete endpoints return 404 rather than ignoring a missing database,
# so check first to keep this cell re-runnable.
databases = [database.name for database in client.raw.databases.list(limit=None)]
database_exists = RAW_DATABASE in databases
print(f"RAW: {RAW_DATABASE}.{RAW_TABLE} ({'present' if database_exists else 'already gone'})")

if CONFIRM and database_exists:
    tables = [table.name for table in client.raw.tables.list(RAW_DATABASE, limit=None)]
    if RAW_TABLE in tables:
        client.raw.tables.delete(RAW_DATABASE, RAW_TABLE)
    client.raw.databases.delete(RAW_DATABASE)
    print("  deleted RAW table and database")

RAW: psp_integrations.pump_readings (present)
  deleted RAW table and database


## 6. Integration

Deleting the integration removes its configuration revisions, task history and errors.

In [17]:
integration = ops.get_integration(client)
print(f"Integration: {integration['externalId'] if integration else 'not found'}")

if CONFIRM and integration:
    client.post(
        f"/api/v1/projects/{client.config.project}/integrations/delete",
        json={"items": [{"externalId": INTEGRATION_EXTERNAL_ID}]},
        headers=ops.BETA,
    )
    print(f"  deleted integration {INTEGRATION_EXTERNAL_ID}")

Integration: psp_pump_feed
  deleted integration psp_pump_feed


## 7. Local files

The feed CSV, the extractor's state file and the saved copy of the last good config.

The state file records how far the extractor has read, so it has to go together with the CSV.
Otherwise a fresh run would skip rows it thinks it has already sent.

In [18]:
from pathlib import Path

from helpers import extractor_config
from helpers.constants import FEED_CSV

state_file = Path("..") / "extractor" / "state.json"
saved_config = extractor_config.GOOD_CONFIG_BACKUP
print(f"Feed CSV   : {FEED_CSV} (exists: {FEED_CSV.exists()})")
print(f"State file : {state_file.resolve()} (exists: {state_file.exists()})")

if CONFIRM:
    FEED_CSV.unlink(missing_ok=True)
    state_file.unlink(missing_ok=True)
    saved_config.unlink(missing_ok=True)
    import time

    time.sleep(3)
    if FEED_CSV.exists():
        print("  The feed CSV was written again - the feed is still running in integrations_tutorial.ipynb.")
        print("  Press Stop feed there (or restart that notebook's kernel), then run this cell again.")
    else:
        print("  deleted local files")

Feed CSV   : C:\Repos\psp-integrations-getting-started\data\pump_readings.csv (exists: True)
State file : C:\Repos\psp-integrations-getting-started\extractor\state.json (exists: True)
  deleted local files


## 8. Capabilities (optional, off by default)

Step 2 of `integrations_tutorial.ipynb` added `integrationsAcl` and
`integrationConfigsAcl` to the Toolkit's service principal group. They are left alone
here on purpose: other work in the same project may now depend on them, and removing
them would break it.

Only revoke them if this project was created solely for this walkthrough. There is no
undo beyond re-running Step 2.

In [19]:
REVOKE_CAPABILITIES = False  # leave False unless you are sure

if CONFIRM and REVOKE_CAPABILITIES:
    from helpers.constants import TOOLKIT_GROUP_NAME

    project = client.config.project
    groups = client.get(f"/api/v1/projects/{project}/groups?all=true").json()["items"]
    group = next(g for g in groups if g["name"] == TOOLKIT_GROUP_NAME)
    kept = [
        c for c in group["capabilities"]
        if not ({"integrationsAcl", "integrationConfigsAcl"} & set(c))
    ]
    body = {"name": group["name"], "capabilities": kept}
    if group.get("sourceId"):
        body["sourceId"] = group["sourceId"]
    elif group.get("members"):
        body["members"] = group["members"]
    created = client.post(f"/api/v1/projects/{project}/groups", json={"items": [body]}).json()["items"][0]
    client.post(f"/api/v1/projects/{project}/groups/delete", json={"items": [group["id"]]})
    print(f"  revoked - group now has {len(created['capabilities'])} capabilities")
else:
    print("Capabilities left in place.")

Capabilities left in place.


## Done

Run `integrations_tutorial.ipynb` again from Step 3 to put everything back.